# Part 4 - Homework 2

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 3) Implement an algorithm computing the next-step solution (x', y', s', µ') from (x, y, s, µ)

In [3]:
def interior_point_step(A, b, c, x, y, s, mu):
    """
    Computes the next iteration step (x', y', s', mu') for the interior-point method.
    
    Parameters:
    A (np.ndarray): constraint matrix of shape (n, m)
    b (np.ndarray): right hand side vector of shape (n,)
    c (np.ndarray): cost vector of shape (m,)
    x, y, s (np.ndarray): current solution vectors
    mu (float): current duality measure parameter
    
    Returns:
    tuple: (x_prime, y_prime, s_prime, mu_prime)
    """
    # 1. Get dimensions
    n, m = A.shape
    
    # 2. Update mu' according to the specific problem parameter
    mu_prime = (1.0 - 1.0 / (6.0 * np.sqrt(m))) * mu
    
    # 3. Set up helper matrices and vectors
    e = np.ones(m)
    
    # S^-1 is a diagonal matrix of 1/s_i
    S_inv = np.diag(1.0 / s)
    
    # X is a diagonal matrix of x_i
    X = np.diag(x)
    
    # D = X * S^-1 is a diagonal matrix with entries x_i / s_i
    D = np.diag(x / s)
    
    # 4. Calculate the step for y (delta_y or 'k')
    # We solve the linear system: (A * D * A^T) * delta_y = b - mu' * A * S^-1 * e
    LHS = A @ D @ A.T
    RHS = b - mu_prime * A @ (S_inv @ e)
    
    delta_y = np.linalg.solve(LHS, RHS)
    
    # 5. Calculate the step for s (delta_s or 'f')
    delta_s = -A.T @ delta_y
    
    # 6. Calculate the step for x (delta_x or 'h')
    delta_x = -X @ S_inv @ delta_s + mu_prime * (S_inv @ e) - x
    
    # 7. Apply the updates
    x_prime = x + delta_x
    y_prime = y + delta_y
    s_prime = s + delta_s
    
    return x_prime, y_prime, s_prime, mu_prime

**TEST**

In [4]:
# 1. Define a simple test system: 1 constraint (n=1), 2 variables (m=2)
A = np.array([[1.0, 1.0]])
b = np.array([2.0])
c = np.array([1.0, 2.0])

# 2. Provide a valid strictly positive starting point
# x satisfies A @ x = b (1*1 + 1*1 = 2) and x > 0
x = np.array([1.0, 1.0])

# y can be unconstrained, let's just use 0
y = np.array([0.0])

# s satisfies A.T @ y + s = c. Since y=0, s must equal c. 
# s = [1, 2], which satisfies s > 0
s = np.array([1.0, 2.0])

# 3. Choose a starting duality measure
mu = 1.0

# 4. Run your implementation
x_prime, y_prime, s_prime, mu_prime = interior_point_step(A, b, c, x, y, s, mu)

# 5. Output the results
print("--- Next Iteration Step ---")
print(f"x':  {x_prime}")
print(f"y':  {y_prime}")
print(f"s':  {s_prime}")
print(f"mu': {mu_prime}")

--- Next Iteration Step ---
x':  [1.33333333 0.66666667]
y':  [0.45118446]
s':  [0.54881554 1.54881554]
mu': 0.882148869802242


## 8) Iterate your next-step algorithm until it converges (or at least stabilizes on most of the digits).

In [14]:
# 1. Load the data from Problem X and Task 6
A = np.array([
    [3.0, 3.0, 3.0, 0.0, 0.0],
    [3.0, 1.0, 0.0, 1.0, 0.0],
    [1.0, 4.0, 0.0, 0.0, 1.0]
])
b = np.array([4.0, 3.0, 4.0])
c = np.array([-3.0, -4.0, 0.0, 0.0, 0.0])

x = np.array([2/5, 8/15, 2/5, 19/15, 22/15])
y = np.array([-4/5, -4/5, -2/3])
s = np.array([37/15, 28/15, 12/5, 4/5, 2/3])

# From Task 7, we know 1.0 is a good starting mu
mu = 1.0 

# 2. Define a tolerance for convergence (how close to 0 mu needs to be)
tolerance = 1e-6
iteration = 0

print(f"Starting mu: {mu}")

# 3. Iterate until mu is smaller than our tolerance
while mu > tolerance:
    # Call the function from Task 3
    x, y, s, mu = interior_point_step(A, b, c, x, y, s, mu)
    iteration += 1
    
    # Optional: Print progress every 10 iterations to watch it stabilize
    if iteration % 10 == 0:
        print(f"Iteration {iteration} | mu: {mu:.8f}")

# 4. Print the final stabilized results
print("-" * 30)
print(f"Converged after {iteration} iterations!")
print(f"Final mu: {mu:.8e}")
print("\nOptimal Primal Solution (x):")
print(np.round(x, 4))
print("\nOptimal Dual Solution (y):")
print(np.round(y, 4))
print("\nOptimal Dual Solution (s):")
print(np.round(s, 4))

Starting mu: 1.0
Iteration 10 | mu: 0.46088988
Iteration 20 | mu: 0.21241949
Iteration 30 | mu: 0.09790199
Iteration 40 | mu: 0.04512204
Iteration 50 | mu: 0.02079629
Iteration 60 | mu: 0.00958480
Iteration 70 | mu: 0.00441754
Iteration 80 | mu: 0.00203600
Iteration 90 | mu: 0.00093837
Iteration 100 | mu: 0.00043249
Iteration 110 | mu: 0.00019933
Iteration 120 | mu: 0.00009187
Iteration 130 | mu: 0.00004234
Iteration 140 | mu: 0.00001951
Iteration 150 | mu: 0.00000899
Iteration 160 | mu: 0.00000415
Iteration 170 | mu: 0.00000191
------------------------------
Converged after 179 iterations!
Final mu: 9.51457656e-07

Optimal Primal Solution (x):
[0.4444 0.8889 0.     0.7778 0.    ]

Optimal Dual Solution (y):
[-0.8889 -0.     -0.3333]

Optimal Dual Solution (s):
[0.     0.     2.6667 0.     0.3333]


## 9) Can you speed up the convergence? How does µ' relate to µ? Can you do better? Can you choose a smaller µ', possibly adaptively, so that the iterative invariants are still satisfied?

So the relationship is strict geometric decrease:
$$\mu' = (1 - \delta)\mu$$
In our example this is $\delta = \frac{1}{6\sqrt{m}}$.

$\mu$ is directly proportional to the duality gap (the difference between the primal and dual objective values). Because $\mu'$ is just a fraction of $\mu$, the duality gap is guaranteed to shrink by that exact same fraction in every single iteration.

The reduction factor $\delta = \frac{1}{6\sqrt{m}}$ is a conservative, worst-case theoretical bound. It was chosen to guarantee that if we take that step, the critical neighborhood invariant ($\sigma^2 \le 1/4$) will always hold.  However, in reality, the actual step usually leaves the variance much smaller than 0.25. By using the worst-case formula, the algorithm is taking tiny steps when it could go faster.

To speed up convergence, we can choose a smaller $\mu'$ adaptively in each iteration, as long as we enforce 3 invariants, so $x' > 0$, $s' > 0$, and the neighborhood constraint remains satisfied, so:
$$\sum_{i=1}^m \left(\frac{x_i' s_i'}{\mu'} - 1\right)^2 \le 0.25$$

We can adapt $\mu'$ with line search by treating it as a dynamic variable - we can push the algorithm to its limits.

In [6]:
def adaptive_interior_point_step(A, b, c, x, y, s, mu):
    """
    Computes the next iteration step using an adaptive line search for mu.
    """
    n, m = A.shape
    
    # Pre-compute static matrices that don't depend on mu'
    e = np.ones(m)
    S_inv = np.diag(1.0 / s)
    X = np.diag(x)
    D = np.diag(x / s)
    
    # LHS of the system remains constant regardless of our mu' choice
    LHS = A @ D @ A.T
    
    # 1. Define our safe, theoretical fallback (worst-case scenario)
    safe_factor = 1.0 - 1.0 / (6.0 * np.sqrt(m))
    mu_safe = safe_factor * mu
    
    # 2. Define our highly aggressive initial guess (e.g., shrink by 80%)
    mu_prime = 0.2 * mu
    
    max_attempts = 10
    
    for attempt in range(max_attempts):
        # Calculate the prospective step using the current candidate mu_prime
        RHS = b - mu_prime * A @ (S_inv @ e)
        delta_y = np.linalg.solve(LHS, RHS)
        delta_s = -A.T @ delta_y
        delta_x = -X @ S_inv @ delta_s + mu_prime * (S_inv @ e) - x
        
        # Calculate prospective new values
        x_new = x + delta_x
        y_new = y + delta_y
        s_new = s + delta_s
        
        # 3. Check the Invariants
        # Condition A: Strict Positivity
        if np.all(x_new > 0) and np.all(s_new > 0):
            # Condition B: The Neighborhood Invariant (sigma^2 <= 0.25)
            sigma_sq = np.sum(((x_new * s_new) / mu_prime - 1.0)**2)
            
            if sigma_sq <= 0.25:
                # Success! The aggressive step works. 
                return x_new, y_new, s_new, mu_prime
        
        # 4. If we reach here, the step failed. Back off by moving mu_prime 
        # halfway closer to the guaranteed safe value.
        mu_prime = (mu_prime + mu_safe) / 2.0
        
        # If our adaptive mu gets too close to the safe mu, just use the safe one
        if mu_safe - mu_prime < 1e-6:
            mu_prime = mu_safe

    # 5. Fallback: If the loop finishes without success, return the theoretically safe step
    RHS = b - mu_safe * A @ (S_inv @ e)
    delta_y = np.linalg.solve(LHS, RHS)
    delta_s = -A.T @ delta_y
    delta_x = -X @ S_inv @ delta_s + mu_safe * (S_inv @ e) - x
    
    return x + delta_x, y + delta_y, s + delta_s, mu_safe

In [ ]:
A = np.array([
    [3.0, 3.0, 3.0, 0.0, 0.0],
    [3.0, 1.0, 0.0, 1.0, 0.0],
    [1.0, 4.0, 0.0, 0.0, 1.0]
])
b = np.array([4.0, 3.0, 4.0])
c = np.array([-3.0, -4.0, 0.0, 0.0, 0.0])

x = np.array([2/5, 8/15, 2/5, 19/15, 22/15])
y = np.array([-4/5, -4/5, -2/3])
s = np.array([37/15, 28/15, 12/5, 4/5, 2/3])

# From Task 7, we know 1.0 is a good starting mu
mu = 1.0 
iteration = 0
tolerance = 1e-6

print("Running ADAPTIVE interior point method...")

while mu > tolerance:
    x, y, s, mu = adaptive_interior_point_step(A, b, c, x, y, s, mu)
    iteration += 1
    
    if iteration % 2 == 0:  # Printing more frequently since it's faster
        print(f"Iteration {iteration} | mu: {mu:.8f}")

print("-" * 30)
print(f"Converged after {iteration} iterations!")
print(f"Final mu: {mu:.8e}")

Running ADAPTIVE interior point method...
Iteration 2 | mu: 0.31666753
Iteration 4 | mu: 0.10027832
Iteration 6 | mu: 0.01128597
Iteration 8 | mu: 0.00045144
Iteration 10 | mu: 0.00001806
Iteration 12 | mu: 0.00000072
------------------------------
Converged after 12 iterations!
Final mu: 7.22301977e-07


## 10) Heuristically decide when to stop. 

In [13]:
n, m = A.shape
# 1. Initialize our clean, exact x array with zeros
exact_x = np.zeros(m)

# 2. Find the indices where s_i < x_i (This is our set 'B')
# These are the indices where x should be strictly positive.
B_indices = np.where(s < x)[0]

# 3. Extract the A_B submatrix (just the columns of A in set B)
A_B = A[:, B_indices]

# 4. Solve the exact system A_B * x_B = b
# Note: If A_B is square, we use standard solve. If it's overdetermined, we use least squares.
exact_x_B = np.linalg.lstsq(A_B, b, rcond=None)[0]

# 5. Place the exact values back into the full vector
exact_x[B_indices] = exact_x_B

print("x from loop:", np.round(x, 6))
print("Exact snapped x* :", np.round(exact_x, 6))

x from loop: [4.44445e-01 8.88888e-01 0.00000e+00 7.77777e-01 2.00000e-06]
Exact snapped x* : [0.444444 0.888889 0.       0.777778 0.      ]


## 12) Commercial solver

In [15]:
from scipy.optimize import linprog

In [16]:
# 1. Define Problem X exactly as we did before
c = np.array([-3.0, -4.0, 0.0, 0.0, 0.0])

A_eq = np.array([
    [3.0, 3.0, 3.0, 0.0, 0.0],
    [3.0, 1.0, 0.0, 1.0, 0.0],
    [1.0, 4.0, 0.0, 0.0, 1.0]
])

b_eq = np.array([4.0, 3.0, 4.0])

# Explicitly state that all variables x_i >= 0
bounds = [(0, None) for _ in range(5)]

# 2. Solve using the Simplex Method
print("--- SOLVING WITH SIMPLEX METHOD (highs-ds) ---")
res_simplex = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs-ds')
print(f"Status: {res_simplex.message}")
print(f"Optimal Objective Value: {res_simplex.fun:.4f}")
print(f"Optimal x: {np.round(res_simplex.x, 4)}")
print()

# 3. Solve using the Interior-Point Method
print("--- SOLVING WITH INTERIOR-POINT METHOD (highs-ipm) ---")
res_ipm = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs-ipm')
print(f"Status: {res_ipm.message}")
print(f"Optimal Objective Value: {res_ipm.fun:.4f}")
print(f"Optimal x: {np.round(res_ipm.x, 4)}")

--- SOLVING WITH SIMPLEX METHOD (highs-ds) ---
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Optimal Objective Value: -4.8889
Optimal x: [0.4444 0.8889 0.     0.7778 0.    ]

--- SOLVING WITH INTERIOR-POINT METHOD (highs-ipm) ---
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Optimal Objective Value: -4.8889
Optimal x: [0.4444 0.8889 0.     0.7778 0.    ]


We get the same solution as in our implementation, and in problem 11 on pdf as well.